In [ ]:
"""
************************************* READ ME *************************************
PREMISES:
    1. You should save your files as *whatever*destain1.csv
    2. BACKGROUND ROI MUST be the last one measured

I have two different files:
1. destain.csv -> all ROIs have been measured in time so I have them listed.
                  Background ROI is named "background"
2. presynapses-slices.txt -> it has the number of ROIs (hence presynapses detected),
                             and the number of slices of the .tif movie.

Enjoy.
Patrick Cottilli
"""
import sys
import os
import statistics
import plotly.express as px
import plotly.graph_objs as go
import plotly.offline as po
import pandas as pd
import numpy as np
import scipy.stats as ss
import math

import plotly.io as pio
pio.renderers.default = "browser"

def MeanOrMed(list, weights = None, str=''):
    #if there are not enough values to perform the test
    if len(list) < 3:
        value = np.median(list)
        str += ' : median (< 3)'
    elif ss.shapiro(list).pvalue < 0.5:
        value = np.median(list)
        str += ' : median'
    else:
        value = np.average(list, weights = weights)
        str += ' : mean'
    return(value,str)

#merge all presynapses
def Merge_pres(df,column, tit):

    pres = df["Presynapse"].unique()
    dF = {}
    for x in pres:
        filt = df[(df.Presynapse==x)]
        dF[x]= list(filt[column])

    dF_list = []
    for x in pres:
        if len(dF_list) != 0:
            for y in range(len(dF[x])):
                dF_list[y].append(dF[x][y])
            continue
        [dF_list.append([j]) for j in dF[x]]

    y_val = []
    y_val_sem = []
    for x in dF_list:
        y_val.append(MeanOrMed(x)[0])
        
        sd = statistics.stdev(x)
        n = len(x)
        sem = sd/math.sqrt(n)
        y_val_sem.append(sem)
    
    x_val = np.arange(len(y_val))*11
    
    fig1 = go.Figure([
            go.Scatter(
                x = x_val,
                y = y_val,
                mode = 'lines',
                line_color='rgba(0, 0, 0, 1)',
                showlegend = False
            ), 
            go.Scatter(
                x = x_val,
                y = np.add(y_val,y_val_sem),
                line = dict(width=0),
                showlegend = False,
                mode = 'lines',
                fillcolor='rgba(0, 0, 0, 0.5)',
                fill='tonexty'
            ),
            go.Scatter(
                x = x_val,
                y = np.subtract(y_val,y_val_sem),
                line = dict(width=0),
                showlegend = False,
                mode = 'lines',
                fillcolor='rgba(0, 0, 0, 0.5)',
                fill='tonexty'
            ),
    ])
    fig1.update_layout(
            autosize = False, width = 1000, height = 500,
            yaxis_title = 'dF/Fo (AU)',
            xaxis_title = 'Time (s)',
            paper_bgcolor = 'rgba(0,0,0,0)',
            plot_bgcolor = 'rgba(0,0,0,0)',
            font = dict(
                family="Arial",
                size=20,
            )
        )


    #AXIS
    
    fig1.update_yaxes(range = [-0.5, 0.1], linecolor='black')
    fig1.update_xaxes(range = [0, x_val[-1]], linecolor='black')
    fig1.update_yaxes(visible=False)
    fig1.update_xaxes(visible=False)
    fig1.add_shape(
        type="line",
        x0=100, x1=150,   # scale length in data units
        y0=-0.3, y1=-0.3,    # position
        line=dict(color="black", width=3)
    )
    
    fig1.add_annotation(
        x=125, y=-0.3,      # middle of the bar
        text="50 seconds",
        showarrow=False,
        yshift=-10     # move label below line
    )
    fig1.add_shape(
        type="line",
        x0=100, x1=100,
        y0=-0.3, y1=-0.2,
        line=dict(color="black", width=3)
    )
    
    fig1.add_annotation(
        x=85, y=-0.25,
        text="0.1 dF/Fo",
        showarrow=False,
        xshift=-20     # move label left
    )
    
    
    fig1.add_hline(y=0, line_dash="dot",
                  #annotation_text="Jan 1, 2018 baseline",
                  #annotation_position="bottom left"
                  )
    fig1.add_vrect(x0="220", x1="385",
                  fillcolor="grey", opacity=0.25, line_width=0
                  )
    fig1.add_trace(go.Scatter(
        x=[220], y=[0.05],
        line_width = 4,
        line_color = 'black',
        mode= 'text',
        text = 'Stim',
        showlegend=False,
        textposition = 'top right'
    ))
    
    fig1.show()
    #fig1.write_image("Figures/meanormed_"+ tit +".png")
    fig1.write_html("Figures/meanormed_"+ tit +".html")


folder = sys.argv[1]

#Dictionary for all my images with their respective csvs
dic = {}

for subfolder in os.listdir(folder):
    path = os.path.abspath(subfolder)+'/'

    #If there are files, hence no subfolders, skip it
    if '.' in subfolder:
        continue

    dic[path] = []

    for files in os.listdir(subfolder):
        
        if 'destain1.csv' in files:
                dic[path].append(path + files)

        if 'slices1.txt' in files:
            dic[path].append(path + files)


#Clean dictionary from not measured folders
rm = []
for key in dic:
    if len(dic[key]) == 0:
        rm.append(key)
for key in rm:
    del dic[key]


if dic:
    out = open('all_destain_mito.csv', 'w')
    out.write("Number,Name,Presynapse,Area,Mean_intensity,Mean_intensity[-]Background,Fo,dF/Fo,Pure%,Slice\n")
    
else:
    sys.exit("There is no image in the dictionary 'dic'")

if 'Figures' not in os.listdir(folder):
    os.mkdir(os.path.abspath(folder) + '/Figures')

#Image counter
f=0
for image in dic:
    print(image)
    f+=1

    #Take number
    number = str(f)
    #Take name
    name = str(image)

    #Open the presynapses-slices.txt file to look at the presynapses number
    #and the number of slices per image
    for file in dic[image]:
        #First line boolean
        i=1

        if 'slices1.txt' not in file:
            continue

        file = open(file)

        for line in file:
            #Skip first line
            if i:
                i=0
                continue

            line = line.split(' ')

            presynapses = int(line[0])
            slices = int(line[1])
        file.close()

    #List for background values
    back = []
    #Open the destain.csv file with the data
    for file in dic[image]:
        #First line boolean
        i=1

        if 'destain1.csv' not in file:
            continue

        file2 = open(file)

        for line in file2:
            line = line.split(',')

            #Look for background
            if i:
                #Column counter to know the position of background values
                j=0
                for col in line:
                    if col == "Mean(background)":
                        back_ind = j
                        break

                    j+=1

                i=0
                continue

            if float(line[back_ind]) != 0:
                back.append(float(line[back_ind]))
        file2.close()

        #open destain in dataframe mode
        df = pd.read_csv(file, header=0, index_col=0, na_values = ['0'])
        col_names = list(df)


        #The columns reset every 10, meaning every 10 is another presynapse
        #Remember that the first value is at position 2 so it goes 12,22,32 etc.
        for x in range(presynapses):
            #Skip Background
            if col_names[2+9*x].split('(')[-1][0] == "b":
                continue

            #Data from destain
            area = df.loc[slices*x+1:slices*(x+1),"Area("+str(x)+")"]
            area = area.values.tolist()
            mean = df.loc[slices*x+1:slices*(x+1),"Mean("+str(x)+")"]
            mean = mean.values.tolist()

            #Take Fo value
            Fo = []
            frames = np.arange(10)
            if x == 0:
                #Range 10 because that is my marked Fo time
                for r in frames:
                    Fo.append(mean[r] - back[r])

                #take the corresponding lines of the presynapse
                df2 = df.iloc[:slices,:10]

                #count for fo
                val = MeanOrMed(Fo, str = 'Fo')
                Fo = val[0]
                #print(val[1])

                #Write the lines
                for n in range(len(mean)):
                    out.write(number + ',' + name + ',')
                    #write the presynapse number from 'Mean(..)'
                    out.write(col_names[2].split('(')[-1].split(')')[0] + ',')
                    #write Area
                    out.write(str(area[n]) + ',')
                    #write mean
                    out.write(str(mean[n]) + ',')
                    #write mean-back
                    F = mean[n] - back[n]
                    out.write(str(F) + ',')
                    #write Fo
                    out.write(str(Fo) + ',')
                    #write dF/Fo
                    out.write(str((F-Fo)/Fo) + ',')
                    #write Pure%
                    out.write(str(F/Fo*100) + ',')
                    #write slice
                    out.write(str(df2.index.values.tolist()[n]) + '\n')

                continue

            #filter the table with the corresponding presynapse
            df2 = df.iloc[slices*x:slices*(x+1),9*x+1:9*x+10]
            #counter for filtered rows
            j=0
            #Range 10 because that is my marked Fo time
            for r in frames:
                Fo.append(mean[r] - back[r])

            #count for fo
            #Fo = statistics.mean(Fo)
            val = MeanOrMed(Fo, str = 'Fo')
            Fo = val[0]
            #print(val[1])

            for n in range(len(mean)):
                out.write(number + ',' + name + ',')
                #write the presynapse number from 'Mean(..)'
                out.write(col_names[2+9*x].split('(')[-1].split(')')[0] + ',')
                #write Area
                out.write(str(area[n]) + ',')
                #write mean
                out.write(str(mean[n]) + ',')
                #write mean-back
                F = mean[n] - back[n]
                out.write(str(F) + ',')
                #write Fo
                out.write(str(Fo) + ',')
                #write dF/Fo
                out.write(str((F-Fo)/Fo)+ ',')
                #write Pure%
                out.write(str(F/Fo*100) + ',')
                #write slice
                out.write(str(int(df2.index.values.tolist()[n]-slices*x)) + '\n')
out.close()

df = pd.read_csv('all_destain_mito.csv', header=0, index_col=False)

#Group by presynapse
#grouped = df.groupby("Presynapse")
#Filter by negative mean of dF/Fo
#filt = grouped.filter(lambda x: x['dF/Fo'].mean() < 0)
#fig = px.line(filt, x="Slice", y="dF/Fo", color="Presynapse")
images = list(df["Number"].unique())
plot = {}
plot_corr = {}
for x in images:
    one_only = df.query("Number ==" + str(x))
    
    fig = px.line(df, x="Slice", y="dF/Fo", color="Presynapse")
    fig.write_html("Figures/all-pres-raw.html")
    #fig.write_image("Figures/all-pres-raw.png")
    
    plot["Presynapse"] = one_only["Presynapse"]
    plot_corr["Presynapse"] = one_only["Presynapse"]
    plot["Slice"] = one_only["Slice"]
    plot_corr["Slice"] = one_only["Slice"]
    plot["dF-corr"] = []
    plot_corr["Corr"] = []
    pres = list(df["Presynapse"].unique())
    slices = list(one_only["Slice"].unique())
    i = -1#pres counter
    for pre in pres:
        i+=1
        pre_one_only = one_only[one_only["Presynapse"] == pre]
        before_stim = pre_one_only[:20]
        xi = np.emath.logn(1/2,before_stim["Slice"])
        yi = before_stim['dF/Fo']#Log1/2(y)
        pol_before_stim = np.polynomial.polynomial.Polynomial.fit(xi,yi,1)

        for sl in slices:
            
            correction = pol_before_stim(np.emath.logn(1/2,sl))

            if np.iscomplexobj(correction):
                correction = float(correction)
            
            plot_corr["Corr"].append(correction)
            ind = sl-1 + len(slices)*i
            plot["dF-corr"].append(pre_one_only["dF/Fo"][ind] - correction)

    df_plot = pd.DataFrame.from_dict(plot)
    fig = px.line(df_plot, x="Slice", y="dF-corr", color="Presynapse")
    fig.show()
    #fig.write_image("Figures/all-pres-corrected.png")
    fig.write_html("Figures/all-pres-corrected.html")
    
    df_corr = pd.DataFrame.from_dict(plot_corr)
    fig1 = px.line(df_corr, x="Slice", y="Corr", color="Presynapse")
    fig1.show()
    fig1.write_html("Figures/Correction-functions.html")
    #fig1.write_image("Figures/Correction-functions.png")
    
    #threshold destain and not
    des = []
    no_des = []
    for pres in df_plot["Presynapse"].unique():
        pre_df_plot = df_plot[df_plot["Presynapse"] == pres]
        
        #eliminate outliers

        
        if pre_df_plot["dF-corr"][20:][pre_df_plot["dF-corr"] < -0.2].count() >= 4 and pre_df_plot["dF-corr"][30:][pre_df_plot["dF-corr"] > -0.2].count() < 3:
            des.append(pres)
            continue
        
        no_des.append(pres)

tot_pres = len(des) + len(no_des)
print(f"The destaining presynapses are: {len(des)/tot_pres*100}%.")
out1 = open(f"destaining-{len(des)/tot_pres*100}%.txt", "w")

df_des = df_plot[df_plot["Presynapse"].isin(des)]
df_nodes = df_plot[df_plot["Presynapse"].isin(no_des)]
fig2 = px.line(df_des, x="Slice", y="dF-corr", color="Presynapse")
fig2.show()

fig2.write_html("Figures/des_pres.html")
#fig2.write_image("Figures/des_pres.png")
fig3 = px.line(df_nodes, x="Slice", y="dF-corr", color="Presynapse")
fig3.write_html("Figures/nodes_pres.html")
#fig3.write_image("Figures/nodes_pres.png")

Merge_pres(df,'dF/Fo',"all-raw")
Merge_pres(df_plot,'dF-corr',"all-corrected")
Merge_pres(df_des, 'dF-corr', "des")
Merge_pres(df_nodes, 'dF-corr', "nodes")